In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf


print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

2025-02-12 20:37:21.795364: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-12 20:37:21.803495: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739392641.812848   27818 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739392641.815622   27818 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-12 20:37:21.825189: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Num GPUs Available:  1


In [2]:
train_file_path = "./train-data.tsv"
test_file_path = "./valid-data.tsv"

In [3]:
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'text'])

In [4]:
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'text'])

In [5]:
train_df.head()

,label,text
0,ham,ahhhh...just woken up!had a bad dream about u ...
1,ham,you can never do nothing
2,ham,"now u sound like manky scouse boy steve,like! ..."
3,ham,mum say we wan to go then go... then she can s...
4,ham,never y lei... i v lazy... got wat? dat day ü ...


$$
\text{Using HuggingFace sentence transformer}
$$
https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=trending

In [6]:
print(train_df.columns)

Index(['label', 'text'], dtype='object')


In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder

sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2', device='cuda')
def preprocess(dataframe):

    texts = dataframe['text'].tolist()

    embeddings = sentence_model.encode(texts, batch_size=64)

    dataframe['encoded'] = list(embeddings)

    label_encoder = LabelEncoder()
    dataframe['label_encoded'] = label_encoder.fit_transform(dataframe['label'])

    dataframe = dataframe.drop(['label', 'text'], axis=1)
    dataframe = dataframe.rename(columns={'label_encoded': 'label', 'encoded': 'text'})
    dataframe.attrs['label_encoder'] = label_encoder

    return dataframe



In [8]:
train = preprocess(train_df)

In [9]:
valid = preprocess(test_df)

In [10]:
train.head()

,text,label
0,"[0.31281447, -0.3585767, 0.0356886, 0.02793775...",0
1,"[0.33434135, -0.088207304, -0.10494735, -0.132...",0
2,"[0.5315699, -0.20541458, 0.15435672, -0.358831...",0
3,"[0.68175876, 0.15692922, -0.017596634, 0.08803...",0
4,"[-0.33380458, 0.043861836, -0.17576396, -0.275...",0


In [11]:
X_train = np.stack(train['text'].values)
y_train = train['label'].values

X_valid = np.stack(valid['text'].values)
y_valid = valid['label'].values

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = Sequential([
    Input(shape=(384,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

I0000 00:00:1739392649.025657   27818 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 449 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


In [13]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=5, batch_size=16)

Epoch 1/5


In [ ]:
def encode(text):
    if text:
        text_encoded = sentence_model.encode(text, convert_to_tensor=False)
        return text_encoded
    else:
        raise ValueError("Text input cannot be empty!")


In [ ]:
def predict_message(pred_text):
    encoded_text = encode(pred_text)
    encoded_text = np.expand_dims(encoded_text, axis=0)

    pred = model.predict(encoded_text, verbose=0)
    pred_value = pred[0][0]

    label = "spam" if pred_value > 0.5 else "ham"

    return [pred_value, label]


In [ ]:
test = predict_message("our new mobile video service is live. just install on your phone to start watching.")

In [ ]:
print(test)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
    test_messages = ["how are you doing today",
                       "sale today! to stop texts call 98912460324",
                       "i dont want to go. can we try it a different day? available sat",
                       "our new mobile video service is live. just install on your phone to start watching.",
                       "you have won £1000 cash! call to claim your prize.",
                       "i'll bring it tomorrow. don't forget the milk.",
                       "wow, is your arm alright. that happened to me one time too"
                      ]

    test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
    passed = True
    output = []

    for msg, ans in zip(test_messages, test_answers):
        prediction = predict_message(msg)
        output.append(prediction[1])

        if prediction[1] != ans:
            passed = False

    print(output)
    if passed:
        print("You passed the challenge. Great job!")
    else:
        print("You haven't passed yet. Keep trying.")

test_predictions()
